# Bias correction of CMIP6 and DestinE forcing against ERA5

This notebook applies statistical bias correction to CMIP6 and DestinE future forcing data, using ERA5 as the observational reference. The model was calibrated on ERA5, so systematic differences between ERA5 and CMIP6/DestinE will propagate into discharge projections unless corrected.

**Approach:**
- **Reference (obs):** ERA5 over the calibration period
- **Simulated historical (simh):** CMIP6 historical / DestinE historical run over the same period
- **Simulated future (simp):** CMIP6 future scenarios / DestinE SSP3-7.0 future run to be corrected

Bias correction is applied per variable using [python-cmethods](https://python-cmethods.readthedocs.io/en/stable/introduction.html):
- `pr` (precipitation): **Quantile Delta Mapping**, multiplicative (`kind='*'`) — preserves future trends in extremes
- `tas` (2m temperature): **Quantile Delta Mapping**, additive (`kind='+'`)
- `rsds` (surface radiation): **Quantile Delta Mapping**, multiplicative (`kind='*'`)

After correction, `evspsblpot` is re-derived via the Makkink method so that it is internally consistent with the corrected meteorology. The bias-corrected forcing is saved to a `bias_corrected/` subdirectory and is ready for use in [step_3b](step_3b_model_run_future.ipynb).

In [ ]:
# General python
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import sys
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import json

# Niceties
from rich import print

# DestinE forcing (Makkink evspsblpot derivation + load methods)
try:
    from scripts.forcing_destine import DestinEForcing, DestinEHistoricalForcing, DestinEFutureForcing
except ImportError:
    project_root = Path().resolve().parent
    sys.path.append(str(project_root))
    from scripts.forcing_destine import DestinEForcing, DestinEHistoricalForcing, DestinEFutureForcing

In [ ]:
# General eWaterCycle
import ewatercycle
import ewatercycle.forcing

In [ ]:
# Bias correction library
from cmethods import adjust

In [ ]:
# Parameters — these get changed when running on HPC
# country = "australia"
# region_id = "camelsaus_102101A"
settings_path = "settings.json"

In [ ]:
# Load settings
with open(settings_path, "r") as json_file:
    settings = json.load(json_file)

display(settings)

## Load ERA5 forcing (reference)

ERA5 is the observational reference on which the HBV model was calibrated. It defines the "true" distribution that CMIP6 and DestinE should be corrected towards.

In [ ]:
# ERA5 forcing is stored in a sub-directory created by eWaterCycle / ESMValTool
era5_load_location = Path(settings['path_ERA5']) / "work" / "diagnostic" / "script"
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=era5_load_location)

display(ERA5_forcing)

In [ ]:
# Load ERA5 variables and restrict to the calibration period
ds_era5 = xr.open_mfdataset([ERA5_forcing['pr'], ERA5_forcing['tas'], ERA5_forcing['rsds']])

cal_start = settings['calibration_start_date'].replace('T00:00:00Z', '')
cal_end   = settings['calibration_end_date'].replace('T00:00:00Z', '')
ds_era5_cal = ds_era5.sel(time=slice(cal_start, cal_end))

print(f"ERA5 calibration window: {cal_start} → {cal_end}")
display(ds_era5_cal)

In [ ]:
# Quick overview plot
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
ds_era5_cal['pr'].plot(ax=axes[0], label='pr')
ds_era5_cal['tas'].plot(ax=axes[1], label='tas')
ds_era5_cal['rsds'].plot(ax=axes[2], label='rsds')
for ax in axes:
    ax.legend()
fig.suptitle('ERA5 reference — calibration period')
plt.tight_layout()

## Bias correction configuration

We use **Quantile Delta Mapping (QDM)** from [python-cmethods](https://python-cmethods.readthedocs.io/en/stable/introduction.html). QDM corrects the quantile distribution of simulated data while preserving the climate-change signal (the delta between historical and future quantiles), making it well-suited for impact studies.

| Variable | Method | `kind` | Rationale |
|---|---|---|---|
| `pr` | QDM | `*` (multiplicative) | Precipitation is bounded at 0; ratios are physically meaningful |
| `tas` | QDM | `+` (additive) | Temperature changes are expressed as absolute differences |
| `rsds` | QDM | `*` (multiplicative) | Radiation is non-negative; ratio-based correction is appropriate |

In [ ]:
BC_CONFIG = {
    'pr':   {'kind': '*', 'n_quantiles': 250},
    'tas':  {'kind': '+', 'n_quantiles': 250},
    'rsds': {'kind': '*', 'n_quantiles': 250},
}
METHOD = 'quantile_delta_mapping'

In [ ]:
def bias_correct_dataset(ds_obs, ds_simh, ds_simp, method=METHOD, config=BC_CONFIG):
    """
    Apply bias correction to each variable in ds_simp.

    Parameters
    ----------
    ds_obs : xr.Dataset
        Observational reference (ERA5) over the training period.
    ds_simh : xr.Dataset
        Simulated historical over the same training period.
    ds_simp : xr.Dataset
        Simulated future data to be corrected.
    method : str
        python-cmethods method name.
    config : dict
        Per-variable settings (kind, n_quantiles).

    Returns
    -------
    xr.Dataset
        Bias-corrected version of ds_simp.
    """
    corrected_vars = {}

    for var, cfg in config.items():
        if var not in ds_obs or var not in ds_simh or var not in ds_simp:
            print(f"[yellow]Skipping {var} — not present in all datasets[/yellow]")
            continue

        # python-cmethods expects 1-D DataArrays (time,) for lumped catchments
        obs  = ds_obs[var].squeeze()
        simh = ds_simh[var].squeeze()
        simp = ds_simp[var].squeeze()

        # Align training period: obs and simh must share the same time axis
        common_time = obs.time.values[
            np.isin(obs.time.values, simh.time.values)
        ]
        obs_aligned  = obs.sel(time=common_time)
        simh_aligned = simh.sel(time=common_time)

        corrected = adjust(
            obs=obs_aligned,
            simh=simh_aligned,
            simp=simp,
            method=method,
            kind=cfg['kind'],
            n_quantiles=cfg['n_quantiles'],
        )

        corrected.attrs = ds_simp[var].attrs
        corrected_vars[var] = corrected

    return xr.Dataset(corrected_vars)

## CMIP6

### Load CMIP6 historical and future forcing

For each dataset/ensemble combination we load:
1. The **historical** run (used together with ERA5 to train the bias correction)
2. Each **future scenario** run (the data that will be corrected)

In [ ]:
def load_cmip_forcing(settings, dataset, experiment, ensemble):
    """Load a LumpedMakkinkForcing object for one CMIP6 dataset/experiment/ensemble."""
    path = (
        Path(settings['path_CMIP6'])
        / dataset / experiment / ensemble
        / "work" / "diagnostic" / "script"
    )
    return ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=path)


def forcing_to_dataset(forcing_obj):
    """Open pr, tas, rsds from a LumpedMakkinkForcing as a single xarray Dataset."""
    return xr.open_mfdataset([forcing_obj['pr'], forcing_obj['tas'], forcing_obj['rsds']])

In [ ]:
# Load CMIP6 historical — first ensemble member of the first dataset, matching HBV calibration
cmip_dataset = settings['CMIP_info']['dataset'][0]
cmip_hist_ensemble = settings['CMIP_info']['ensembles'][0]
forcing_obj = load_cmip_forcing(settings, cmip_dataset, 'historical', cmip_hist_ensemble)
ds_cmip_hist = forcing_to_dataset(forcing_obj).sel(time=slice(cal_start, cal_end))

print(f"Loaded CMIP6 historical: {cmip_dataset} / {cmip_hist_ensemble}")
display(ds_cmip_hist)

In [ ]:
# Load CMIP6 future for every dataset × experiment × ensemble combination
cmip_future = {}  # key: (dataset, experiment, ensemble) → xr.Dataset

for dataset in settings['CMIP_info']['dataset']:
    for experiment in settings['CMIP_info']['experiments'][1:]:  # skip 'historical'
        for ensemble in settings['CMIP_info']['ensembles']:
            key = (dataset, experiment, ensemble)
            try:
                forcing_obj = load_cmip_forcing(settings, dataset, experiment, ensemble)
                cmip_future[key] = forcing_to_dataset(forcing_obj)
                print(f"Loaded: {dataset} / {experiment} / {ensemble}")
            except FileNotFoundError:
                print(f"[yellow]Skipping (not found): {dataset} / {experiment} / {ensemble}[/yellow]")

### Apply bias correction to CMIP6 future

In [ ]:
bc_results = {}  # (dataset, experiment, ensemble) → xr.Dataset

for (dataset, experiment, ensemble), ds_simp in cmip_future.items():
    print(f"\nBias correcting: {dataset} / {experiment} / {ensemble}")
    ds_corrected = bias_correct_dataset(
        ds_obs=ds_era5_cal,
        ds_simh=ds_cmip_hist,
        ds_simp=ds_simp,
    )
    bc_results[(dataset, experiment, ensemble)] = ds_corrected
    print(f"  Done")

### Re-derive evspsblpot for CMIP6

The `evspsblpot` stored in the raw CMIP6 forcing was computed from the original (biased) `tas` and `rsds`. After correcting those variables, `evspsblpot` must be recomputed to remain consistent with the corrected meteorology — HBV uses it directly.

In [ ]:
for key in bc_results:
    bc_results[key] = DestinEForcing.derive_e_pot(bc_results[key])
    print(f"evspsblpot derived for CMIP6 {key}")

### Save CMIP6 bias-corrected forcing

Corrected NetCDF files are written to a `bias_corrected/` subfolder alongside the raw CMIP6 forcing, together with a `config.json` so the forcing can be loaded as a `LumpedMakkinkForcing` in subsequent notebooks.

In [ ]:
from datetime import datetime


def save_cmip_bc_forcing(ds_bc, settings, dataset, experiment, ensemble):
    """Save bias-corrected CMIP6 forcing to a bias_corrected/ subfolder.

    Writes NetCDF files and an ewatercycle_forcing.yaml so the result can be
    loaded with LumpedMakkinkForcing.load(directory=...) in step_3b by simply
    pointing load_location to the bias_corrected/ folder instead of
    work/diagnostic/script/.
    """
    out_dir = (
        Path(settings['path_CMIP6'])
        / dataset / experiment / ensemble
        / "bias_corrected"
    )
    out_dir.mkdir(parents=True, exist_ok=True)

    future_start = settings['future_start_date']
    future_end   = settings['future_end_date']
    start_str = datetime.strptime(future_start, "%Y-%m-%dT%H:%M:%SZ").strftime("%Y_%m_%d")
    end_str   = datetime.strptime(future_end,   "%Y-%m-%dT%H:%M:%SZ").strftime("%Y_%m_%d")
    name      = f"CMIP6_BC_{dataset}_{experiment}_{ensemble}"

    files = {
        'pr':         str(out_dir / f"{name}_pr_{start_str}-{end_str}.nc"),
        'tas':        str(out_dir / f"{name}_tas_{start_str}-{end_str}.nc"),
        'rsds':       str(out_dir / f"{name}_rsds_{start_str}-{end_str}.nc"),
        'evspsblpot': str(out_dir / f"{name}_evspsblpot_{start_str}-{end_str}.nc"),
    }

    ds_save = ds_bc.copy()
    ds_save['time'] = pd.to_datetime(ds_save['time'].values).tz_localize(None)
    for var, path in files.items():
        if var in ds_save:
            ds_save[var].to_netcdf(path)

    # Write ewatercycle_forcing.yaml so step_3b can use .load() without modification
    forcing_obj = ewatercycle.forcing.sources["LumpedMakkinkForcing"](
        directory=str(out_dir),
        start_time=future_start,
        end_time=future_end,
        shape=settings['path_shape'],
        filenames=files,
    )
    forcing_obj.save(out_dir)

    return out_dir


cmip_saved_paths = {}

for (dataset, experiment, ensemble), ds_bc in bc_results.items():
    out_dir = save_cmip_bc_forcing(ds_bc, settings, dataset, experiment, ensemble)
    cmip_saved_paths[(dataset, experiment, ensemble)] = out_dir
    print(f"Saved → {out_dir}")

## DestinE

### Load DestinE historical and future forcing

The DestinE historical simulation (IFS-FESOM, ~1990–2020) is used as `simh` to train the bias correction. The DestinE SSP3-7.0 future simulation is the `simp` to be corrected.

In [ ]:
# DestinE historical (simh)
DestinE_hist_forcing = DestinEHistoricalForcing.load(settings["path_DestinE_historical"])
ds_destine_hist = xr.open_mfdataset([
    DestinE_hist_forcing['pr'], DestinE_hist_forcing['tas'], DestinE_hist_forcing['rsds']
])
ds_destine_hist_cal = ds_destine_hist.sel(time=slice(cal_start, cal_end))

print(f"DestinE historical calibration window: {ds_destine_hist_cal.time.values[[0, -1]]}")
display(ds_destine_hist_cal)

In [ ]:
# DestinE future (simp)
DestinE_future_forcing = DestinEFutureForcing.load(settings["path_DestinE"])
ds_destine_future = xr.open_mfdataset([
    DestinE_future_forcing['pr'], DestinE_future_forcing['tas'], DestinE_future_forcing['rsds']
])

display(ds_destine_future)

### Apply bias correction to DestinE future

In [ ]:
print("Bias correcting DestinE future...")
ds_destine_corrected = bias_correct_dataset(
    ds_obs=ds_era5_cal,
    ds_simh=ds_destine_hist_cal,
    ds_simp=ds_destine_future,
)
print("Done")
display(ds_destine_corrected)

### Re-derive evspsblpot for DestinE

In [ ]:
ds_destine_corrected = DestinEForcing.derive_e_pot(ds_destine_corrected)
print("evspsblpot derived for DestinE future")

### Save DestinE bias-corrected forcing

In [ ]:
# Read start/end from the config.json written by DestinEFutureForcing.generate()
destine_config_path = Path(settings['path_DestinE']) / "config.json"
with open(destine_config_path, "r") as f:
    destine_config = json.load(f)

destine_start = destine_config['start']
destine_end   = destine_config['end']
start_str = datetime.strptime(destine_start, "%Y-%m-%dT%H:%M:%SZ").strftime("%Y_%m_%d")
end_str   = datetime.strptime(destine_end,   "%Y-%m-%dT%H:%M:%SZ").strftime("%Y_%m_%d")

destine_bc_dir = Path(settings['path_DestinE']) / "bias_corrected"
destine_bc_dir.mkdir(parents=True, exist_ok=True)

destine_files = {
    'pr':         str(destine_bc_dir / f"DestinE_BC_pr_{start_str}-{end_str}.nc"),
    'tas':        str(destine_bc_dir / f"DestinE_BC_tas_{start_str}-{end_str}.nc"),
    'rsds':       str(destine_bc_dir / f"DestinE_BC_rsds_{start_str}-{end_str}.nc"),
    'evspsblpot': str(destine_bc_dir / f"DestinE_BC_evspsblpot_{start_str}-{end_str}.nc"),
}

ds_destine_save = ds_destine_corrected.copy()
ds_destine_save['time'] = pd.to_datetime(ds_destine_save['time'].values).tz_localize(None)
for var, path in destine_files.items():
    if var in ds_destine_save:
        ds_destine_save[var].to_netcdf(path)

# Write ewatercycle_forcing.yaml so step_3b can load DestinE BC with .load()
destine_bc_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"](
    directory=str(destine_bc_dir),
    start_time=destine_start,
    end_time=destine_end,
    shape=destine_config['shape'],
    filenames=destine_files,
)
destine_bc_forcing.save(destine_bc_dir)

print(f"Saved DestinE bias-corrected forcing → {destine_bc_dir}")

## Diagnostic plots

Compare the raw and bias-corrected distributions against ERA5 for each variable (monthly climatology and CDF).

In [ ]:
def plot_climatology_comparison(ds_obs, ds_raw, ds_corrected, variable, title_suffix=""):
    """Plot monthly mean climatology and CDF for ERA5, raw, and bias-corrected data."""
    common_start = max(ds_obs.time.values.min(), ds_raw.time.values.min())
    common_end   = min(ds_obs.time.values.max(), ds_raw.time.values.max())

    obs_clim = ds_obs[variable].sel(time=slice(common_start, common_end)).groupby('time.month').mean()
    raw_clim = ds_raw[variable].sel(time=slice(common_start, common_end)).groupby('time.month').mean()
    bc_clim  = ds_corrected[variable].sel(time=slice(common_start, common_end)).groupby('time.month').mean()

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    obs_clim.plot(ax=ax, label='ERA5 (reference)', color='black', linewidth=2)
    raw_clim.plot(ax=ax, label='Raw', color='steelblue', linestyle='--')
    bc_clim.plot(ax=ax, label='Bias-corrected', color='tomato')
    ax.set_title(f'Monthly climatology — {variable}\n{title_suffix}')
    ax.set_xlabel('Month')
    ax.legend()

    ax = axes[1]
    for data, label, color, ls in [
        (ds_obs[variable].sel(time=slice(common_start, common_end)).values.ravel(), 'ERA5', 'black', '-'),
        (ds_raw[variable].sel(time=slice(common_start, common_end)).values.ravel(), 'Raw', 'steelblue', '--'),
        (ds_corrected[variable].sel(time=slice(common_start, common_end)).values.ravel(), 'BC', 'tomato', '-'),
    ]:
        sorted_data = np.sort(data[~np.isnan(data)])
        cdf = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
        ax.plot(sorted_data, cdf, label=label, color=color, linestyle=ls)
    ax.set_title(f'CDF — {variable}')
    ax.set_xlabel(variable)
    ax.set_ylabel('Cumulative probability')
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# CMIP6 diagnostics — first available combination
first_key = next(iter(bc_results))
dataset, experiment, ensemble = first_key

for var in ['pr', 'tas', 'rsds']:
    plot_climatology_comparison(
        ds_obs=ds_era5_cal,
        ds_raw=cmip_future[first_key],
        ds_corrected=bc_results[first_key],
        variable=var,
        title_suffix=f"CMIP6: {dataset} / {experiment} / {ensemble}",
    )

In [ ]:
# DestinE diagnostics
for var in ['pr', 'tas', 'rsds']:
    plot_climatology_comparison(
        ds_obs=ds_era5_cal,
        ds_raw=ds_destine_future,
        ds_corrected=ds_destine_corrected,
        variable=var,
        title_suffix="DestinE SSP3-7.0",
    )

## Verify saved forcing

Reload one CMIP6 and the DestinE bias-corrected forcing as `LumpedMakkinkForcing` objects to confirm the files are valid and ready for use in step_3b.

In [ ]:
# Verify CMIP6 — load using the same pattern as step_3b, just pointing to bias_corrected/
first_key = next(iter(cmip_saved_paths))
dataset, experiment, ensemble = first_key
load_location = cmip_saved_paths[first_key]

cmip_bc_check = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=load_location)
display(cmip_bc_check)

# Verify DestinE
destine_bc_check = ewatercycle.forcing.sources["LumpedMakkinkForcing"].load(directory=destine_bc_dir)
display(destine_bc_check)

print("All bias-corrected forcing loaded successfully — ready for use in step_3b.")